# 강의 04 · 실습 6 — 커스텀 MCP 서버 · (6) 고난도 III

## 1. 문제상황

- 쇼핑몰의 상품 정보(가격·재고)는 상품팀이 관리하고, 배송비와 할인 정책은 정책팀이 관리합니다.
- 두 팀은 서로의 코드를 고치지 않기로 했기 때문에, 조회 함수는 두 파이썬 파일에 나뉘어 있습니다.
- 고객이 "키보드 2개 사면 배송비 포함 얼마예요?"라고 물으면 상담원은 상품팀 화면에서 가격을 보고, 정책팀 표에서 배송비를 찾아 더합니다.
- 모델이 답하게 하려면 두 팀의 함수를 각자의 서버로 내놓고, 클라이언트가 두 서버의 도구를 한 목록으로 받아 써야 합니다.

## 2. 문제와 목표

- **문제**: 상품 정보와 정책이 서로 다른 팀의 파일에 있어, 한 질문에 답하려면 두 곳의 함수가 모두 필요합니다.
- **목표**: 서버 두 대를 각각 다른 파일로 만들고, 클라이언트 하나가 두 서버를 함께 띄워 도구 목록을 병합해 받은 뒤, 배송비 포함 가격을 묻는 질문에 두 서버의 도구를 모두 호출해 답하게 합니다.
    - 서버 `catalog`: 가격을 돌려주는 `get_price`와 재고를 돌려주는 `get_stock`을 가집니다. 가격은 무선 마우스 12,000원·키보드 45,000원·모니터 210,000원, 재고는 37·0·5개입니다. 값은 「6. 코드 — 스텝바이스텝」 단계 0에 코드로 주어져 있습니다.
    - 서버 `policy`: 주문 총액을 받아 배송비를 돌려주는 `shipping_fee`를 가집니다. 총액이 5만 원 이상이면 0원, 아니면 3,000원입니다.
    - 서버 파일 첫 줄은 `from mcp.server.fastmcp import FastMCP`입니다.
    - 질문 두 문장(「키보드 2개를 사면 배송비 포함 얼마인가」·「무선 마우스 1개는 배송비 포함 얼마인가」)은 코드에 미리 정해 넣습니다. 도구 목록은 「서버가 준 도구:」 줄로 출력합니다.
- **목표 달성 여부의 판정 기준**
    - 도구 목록에 두 서버의 도구 세 개가 모두 있습니다.
    - 키보드 질문에서 `get_price`가 45000, `shipping_fee`가 0을 돌려주어 최종 답이 90,000원입니다.
    - 무선 마우스 질문에서 `get_price`가 12000, `shipping_fee`가 3000을 돌려주어 최종 답이 15,000원인 것을 실행 결과에서 확인합니다. 모든 도구 호출이 오류 없이 끝납니다.


## 3. 워크플로우 다이어그램

## 4. 단계별 요구사항


## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

클라이언트 쪽 준비입니다. 라이브러리를 불러오고 모델을 준비하고, 도구 호출 루프 `build_loop`와 연결 선언 함수 `server_config`를 정의합니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- `build_loop`는 받아 온 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결하는 도구 호출 루프입니다. 서버 코드가 아니라 서버를 쓰는 쪽의 코드입니다.
- `sys.stderr = sys.__stderr__` 줄은 노트북 전용입니다. 노트북 커널은 표준 오류 스트림을 화면용 객체로 바꿔 두는데, 서버 프로세스를 띄우는 코드는 원래의 표준 오류 스트림을 요구하므로 되돌려 놓습니다. 이 줄은 MCP 클라이언트를 불러오기 전에 있어야 합니다.
- `server_config`는 서버 파일 하나를 표준입출력으로 띄우는 연결 선언입니다. `sys.executable`은 지금 돌고 있는 파이썬 러너입니다. `FASTMCP_LOG_LEVEL`은 서버의 안내 로그가 화면을 채우지 않게 하는 설정입니다.
- 실행 결과 출력은 `show(result)`, 메시지 글자 추출은 `text_of(m)`로 합니다.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

sys.stderr = sys.__stderr__   # 노트북 커널의 stderr에는 fileno()가 없어 서버 프로세스 시작이 실패하므로 원래 stderr로 되돌린다
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")


class State(TypedDict):
    messages: Annotated[list, add_messages]


def build_loop(tools):
    """도구 호출 루프. 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결한다."""
    bound = llm.bind_tools(tools)

    def call_model(state: State) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    def should_continue(state: State) -> str:
        return "tools" if state["messages"][-1].tool_calls else END

    g = StateGraph(State)
    g.add_node("model", call_model)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "model")
    g.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
    g.add_edge("tools", "model")
    return g.compile()


def text_of(m) -> str:
    """메시지 내용이 콘텐츠 블록 목록이면 글자 부분만 이어 붙인다."""
    if isinstance(m.content, list):
        return " ".join(p.get("text", "") for p in m.content if isinstance(p, dict))
    return str(m.content)


def show(result) -> None:
    """실행 결과의 메시지를 종류·도구 호출·상태와 함께 한 줄씩 출력한다."""
    for m in result["messages"]:
        kind = type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            print(f"[{kind}] tool_calls={[(c['name'], c['args']) for c in calls]}")
        elif kind == "ToolMessage":
            print(f"[{kind}] status={m.status!r} {text_of(m)[:160]}")
        elif m.content:
            print(f"[{kind}] {text_of(m)[:300]}")


def server_config(file: str) -> dict:
    """서버 파일 하나를 표준입출력으로 띄우는 연결 선언을 만든다."""
    return {"command": sys.executable, "args": [str(Path(file).resolve())],
            "transport": "stdio", "env": {"FASTMCP_LOG_LEVEL": "ERROR"}}


print("클라이언트 준비를 마쳤습니다.")

# 주어진 자료 — 두 서버 파일(catalog_server.py·policy_server.py)의 가격·재고·배송비 기준. 서버 파일의 도구 등록 코드에 그대로 옮겨 적는다
PRICE_DB = {"무선 마우스": 12000, "키보드": 45000, "모니터": 210000}   # 원 단위 정수
STOCK_DB = {"무선 마우스": 37, "키보드": 0, "모니터": 5}
# 배송비 기준: 주문 총액이 50000 이상이면 0, 아니면 3000 — policy_server.py의 도구 코드 안에 그대로 쓴다


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다. 서버 파일은 `%%writefile` 셀로 만듭니다.


## 7. 실행 결과 확인

위 실행 결과에서 다음을 확인합니다.

1. `서버가 준 도구:` 줄에 `catalog` 서버의 `get_price`·`get_stock`과 `policy` 서버의 `shipping_fee`가 한 목록에 있습니다.
2. 키보드 질문에서 `get_price`(45000)와 `shipping_fee`(0)가 호출되고 최종 답이 90,000원입니다.
3. 무선 마우스 질문에서 `get_price`(12000)와 `shipping_fee`(3000)가 호출되고 최종 답이 15,000원입니다.
4. 모든 `ToolMessage`가 `status='success'`이고, 최종 답의 숫자가 전부 도구 결과에서 온 값입니다.